In [0]:
eilerspath='development_042_silver_sandbox.demand_forecast.eilers_landbased_performance_clean_sl'
mappingpath='development_042_silver_sandbox.demand_forecast.eilers_to_gdna_mapping_silver'
dest='development_042_silver_sandbox.demand_forecast.eilers_group_historical_time_series'

In [0]:
eilers=spark.read.table(eilerspath)
mapping_df = spark.read.table(mappingpath)
display(eilers)

In [0]:
from pyspark.sql import functions as F

joined_df = eilers.join(
    mapping_df,
    eilers.game_name== F.lower(mapping_df.bp_theme_name),
    "inner"
)

display(joined_df)

In [0]:
from pyspark.sql import functions as F

# Filter for U.S. & Canada region
eilers_us = eilers.filter(F.col("region_name") == "U.S. & Canada")

# Define grouping columns
group_cols = ["game_classification", "game_name", "yearmonth", "parent_own_status", "own_status"]

# Columns to sum
sum_cols = ["no_of_slots", "casinos"]

# Columns to weighted-average by no_of_slots
weighted_avg_cols = [
    "avg_average_bet",
    "avg_days_game_theme",
    "avg_days_terminal",
    "avg_games_played_index",
    "avg_coin_in_index_vs_house",
    "avg_theo_net_win_index_vs_house",
    "avg_coin_in_vs_zone",
    "avg_theo_net_win_index_vs_zone",
    "average_bet_index_same_denom",
    "average_bet_index_same_type_same_denom",
    "coin_in_vs_house_same_type",
    "theo_win_vs_house_same_type_same_denom",
    "utilization_index_same_type_same_denom",
]

# Build aggregation expressions
agg_exprs = [F.sum(F.col(c)).alias(c) for c in sum_cols]

# Weighted average: sum(col * no_of_slots) / sum(no_of_slots)
agg_exprs += [
    (F.sum(F.col(c) * F.col("no_of_slots")) / F.sum(F.col("no_of_slots"))).alias(c)
    for c in weighted_avg_cols
]

# Apply grouping and aggregation
eilers_grouped = eilers_us.groupBy(group_cols).agg(*agg_exprs)

# Replace nulls with column means for the aggregated metric columns
mean_values = eilers_grouped.select(
    [F.mean(F.col(c)).alias(c) for c in sum_cols + weighted_avg_cols]
).first().asDict()

eilers_grouped = eilers_grouped.fillna(mean_values)

display(eilers_grouped)

In [0]:
# avg_theo_net_win_index_vs_house avg_coin_in_index_vs_house ( check zone too if not good)
tmp = eilers_grouped.filter(F.col("game_name").contains("king khufu - dawn"))
display(tmp)

In [0]:
eilers_grouped.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(dest)